In [0]:
%sql 
select count(distinct(trip_id))
from rideflow.bronze.trips_raw

In [0]:
%sql
SELECT trip_id, trip_status, rider_rating, fare_inr, event_ts, _source_file
FROM rideflow.bronze.trips_raw
WHERE trip_id = 'T0001747'          
ORDER BY event_ts

In [0]:
%sql
SELECT
  trip_id,
  COUNT(*)                       AS copies,
  COUNT(DISTINCT event_ts)       AS distinct_versions,
  COUNT(DISTINCT _source_file)   AS files
FROM rideflow.bronze.trips_raw
GROUP BY trip_id
HAVING COUNT(*) > 1
ORDER BY distinct_versions DESC

In [0]:
%sql 
select *
from rideflow.bronze.trips_raw
where driver_id is null and trip_status='COMPLETED'

In [0]:
%sql
SELECT COUNT(*) FROM rideflow.bronze.trips_raw
WHERE fare_inr < 0

In [0]:
%sql
SELECT t.city_id, COUNT(*) AS trips
FROM rideflow.bronze.trips_raw t
LEFT JOIN rideflow.bronze.cities c
  ON t.city_id = c.city_id
WHERE c.city_id IS NULL
GROUP BY t.city_id

In [0]:
from pyspark.sql import functions as F 
from pyspark.sql.window import Window 

dbutils.widgets.text("catalog","rideflow")
catalog=dbutils.widgets.get("catalog")

raw=spark.read.table(f"{catalog}.bronze.trips_raw")

print(raw.count())



In [0]:
w=Window.partitionBy("trip_id").orderBy(F.col("event_ts").desc())

latest=(
    raw.withColumn("rn",F.row_number().over(w))
    .filter(F.col("rn")==1)
    .drop("rn")
)

print(latest.count())


In [0]:
ts_cols = ["request_ts", "pickup_ts", "dropoff_ts", "event_ts"]

typed = latest
for c in ts_cols:
    typed = typed.withColumn(c, F.to_timestamp(F.col(c)))          # string → timestamp

typed = typed.withColumn(
    "trip_date_ist",
    F.to_date(F.from_utc_timestamp(F.col("request_ts"), "Asia/Kolkata"))  # UTC → India time
)

typed.printSchema()
typed.select("trip_id", "request_ts", "trip_date_ist").show(5, truncate=False)

In [0]:
valid_cities=[ r.city_id for r in spark.read.table(f"{catalog}.bronze.cities").select("city_id").collect()]
print(valid_cities)

In [0]:
checked = typed.withColumn(
    "failed_rules",
    F.array_compact(F.array(
        F.when( (F.col("trip_status") == "COMPLETED") & F.col("driver_id").isNull(),  F.lit("completed_has_driver")),
        F.when( F.col("fare_inr")< 0,                                                  F.lit("fare_not_negative")),
        F.when( F.col("dropoff_ts")< F.col("pickup_ts"),                               F.lit("dropoff_after_pickup")),
        F.when( ~F.col("city_id").isin(valid_cities),                                 F.lit("valid_city")),
    ))
)

In [0]:
good = checked.filter(F.size("failed_rules") == 0)
bad  = checked.filter(F.size("failed_rules") > 0)
print(good.count(), bad.count())     

display(bad.groupBy("failed_rules").count())

In [0]:
def clean_trips(df):
    # 1. dedupe: latest row per trip_id
    w = Window.partitionBy("trip_id").orderBy(F.col("event_ts").desc())
    df = (df.withColumn("rn", F.row_number().over(w))
            .filter(F.col("rn") == 1)
            .drop("rn"))

    # 2. types + IST date
    for c in ["request_ts", "pickup_ts", "dropoff_ts", "event_ts"]:
        df = df.withColumn(c, F.to_timestamp(F.col(c)))
    df = df.withColumn(
        "trip_date_ist",
        F.to_date(F.from_utc_timestamp(F.col("request_ts"), "Asia/Kolkata")))

    # 3. rules
    df = df.withColumn(
        "failed_rules",
        F.array_compact(F.array(
            F.when((F.col("trip_status") == "COMPLETED") & F.col("driver_id").isNull(), F.lit("completed_has_driver")),
            F.when(F.col("fare_inr") < 0,                                              F.lit("fare_not_negative")),
            F.when(F.col("dropoff_ts") < F.col("pickup_ts"),                           F.lit("dropoff_after_pickup")),
            F.when(~F.col("city_id").isin(valid_cities),                               F.lit("valid_city")),
        ))
    )

    # 4. split
    good = df.filter(F.size("failed_rules") == 0).drop("failed_rules")
    bad  = (df.filter(F.size("failed_rules") > 0)
              .withColumn("_quarantined_at", F.current_timestamp()))
    return good, bad

In [0]:
good, bad = clean_trips(spark.read.table(f"{catalog}.bronze.trips_raw"))
print(good.count(), bad.count())      # must be 4526  36

In [0]:
if not spark.catalog.tableExists(f"{catalog}.silver.trips"):
    good.limit(0).write.saveAsTable(f"{catalog}.silver.trips")

if not spark.catalog.tableExists(f"{catalog}.silver.trips_quarantine"):
    bad.limit(0).write.saveAsTable(f"{catalog}.silver.trips_quarantine")

In [0]:
from delta.tables import DeltaTable

def process_batch(batch_df, batch_id):
    good, bad = clean_trips(batch_df)

    # 1. good rows → MERGE into silver.trips
    target = DeltaTable.forName(spark, f"{catalog}.silver.trips")
    (target.alias("t")
        .merge(good.alias("s"), "t.trip_id = s.trip_id")
        .whenMatchedUpdateAll(condition="s.event_ts > t.event_ts")
        .whenNotMatchedInsertAll()
        .execute())

    # 2. bad rows → append to quarantine
    bad.write.mode("append").saveAsTable(f"{catalog}.silver.trips_quarantine")

    print(f"batch {batch_id}: good={good.count()}, bad={bad.count()}")

In [0]:
chk_root = f"/Volumes/{catalog}/raw/checkpoints"

q = (spark.readStream
        .table(f"{catalog}.bronze.trips_raw")                        # which Bronze table to read from?
      .writeStream
        .foreachBatch(process_batch)                                     # hand each micro-batch to your function
        .option("checkpointLocation", f"{chk_root}/silver_trips")
        .trigger(availableNow=True)
        .start())

q.awaitTermination()

In [0]:
%sql
SELECT (SELECT COUNT(*) FROM rideflow.silver.trips)            AS silver,
       (SELECT COUNT(*) FROM rideflow.silver.trips_quarantine) AS quarantine